# PolOCRBench Constrained — QLoRA Qwen3-VL (subtask A)

Etap 1: przygotowanie (CPU) — repo, oficjalny train z `slayerlabs/PolEvalDS`, corpus pinowany sha.
Etap 2: `--smoke-train` (jeden krok GPU) — bez niego trening zablokowany.
Etap 3: `--execute` — 3 epoki QLoRA, adaptery per epoka, DEV CER/WER z ochrona baseline.
Etap 4: oficjalny scoring DEV scorerem organizatorow (`polocr_eval.py`, 1-CER z CI).
Etap 5: dowody (zip) na Drive.

Ten sam pin modelu i hiperparametry co `runpod_qwen3vl_histfinetune` — wynik porownywalny z seria historyczna.

In [ ]:
# PolOCRBench Constrained: QLoRA Qwen3-VL na oficjalnym trainie (subtask A).
# Wzorzec jak colab_qwen3vl_hist_qlora: przygotowanie -> smoke GPU -> trening -> dowody.
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, subprocess, sys
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Brak GPU. Srodowisko wykonawcze > Zmien typ > GPU i uruchom ponownie.')
print('GPU', torch.cuda.get_device_name(0), 'VRAM_GiB', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))

from google.colab import drive
drive.mount('/content/drive')
work = Path('/content/drive/MyDrive/OCR_engine') / ('poleval-qlora-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
work.mkdir(parents=True, exist_ok=False)
print('WORK', work)

def digest(p):
    with open(p, 'rb') as h:
        return hashlib.file_digest(h, 'sha256').hexdigest()

def run_logged(command, name, cwd=None, env=None):
    log = work / 'logs' / f'{name}.log'
    log.parent.mkdir(parents=True, exist_ok=True)
    with log.open('w', encoding='utf-8') as out:
        process = subprocess.Popen(command, cwd=cwd, env=env or {**os.environ}, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            out.write(line); out.flush(); print(line, end='')
        code = process.wait()
    if code != 0:
        raise RuntimeError(f'{name} failed with exit {code}; log: {log}')
    return log

# 1) Kod: przypiety commit repo (zmien kod = nowy eksperyment, nie restart).
REPO_COMMIT = os.environ.get('OCR_ENGINE_COMMIT', '82352c716f66aa815ccd6279ceb938fdf29cbab1')
repo = Path('/content/OCR_engine')
if not repo.exists():
    run_logged(['git', 'clone', 'https://github.com/PiotrStyla/OCR_engine.git', str(repo)], 'clone-engine')
    run_logged(['git', 'checkout', REPO_COMMIT], 'checkout-engine', cwd=repo)
print('ENGINE_COMMIT', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=repo, text=True).strip())

# 2) Dane: oficjalny PolEvalDS (train), corpus budowany deterministycznie i pinowany sha.
# 2) Dane: prywatny mirror corpusu na HF; token z Secrets Colab (HF_TOKEN) lub srodowiska.
from huggingface_hub import login, snapshot_download
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = os.environ.get('HF_TOKEN')
if not hf_token:
    raise RuntimeError('Brak HF_TOKEN: dodaj go w Secrets Colab (klucz HF_TOKEN) albo w zmiennych srodowiskowych.')
login(token=hf_token)
corpus = Path(snapshot_download('PiotrSty/poleval-corpus-v1', repo_type='dataset',
                                local_dir='/content/poleval-corpus-v1'))
CORPUS_MANIFEST_SHA256 = 'af14901cce55094826cc3caf7a0d2402c89df8e497cd2fedfef446752df74717'
if digest(corpus / 'pages.jsonl') != CORPUS_MANIFEST_SHA256:
    raise RuntimeError('Corpus manifest nie zgadza sie z pinem. Zly mirror albo zmieniony builder.')

# 3) Zaleznosci treningowe (CPU-only torch jest w Colab; dokladamy reszte).
run_logged([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.57', 'peft', 'bitsandbytes', 'jiwer', 'accelerate'], 'pip')
print('PRZYGOTOWANIE_OK')


In [ ]:
# Smoke GPU: jeden krok optymalizatora na najciezszej stronie treningowej.
run_logged([sys.executable, '-u', '-m', 'training.run_qwen3vl_poleval_qlora',
            '--smoke-train', '--corpus', str(corpus),
            '--output', str(work / 'smoke')], 'smoke', cwd=repo)
receipt = json.loads((work / 'smoke' / 'smoke-result.json').read_text(encoding='utf-8'))
if receipt['status'] != 'ok' or receipt.get('cpu_structural_smoke', True) or receipt.get('optimizer_steps') != 1:
    raise RuntimeError('Brak udanego smoke GPU; pelny trening zablokowany.')
print('SMOKE_OK', receipt['page_id'], receipt['peak_reserved_bytes'])


In [ ]:
# Pelny trening: 3 epoki, trwale adaptery per epoka, ochrona baseline DEV.
run_logged([sys.executable, '-u', '-m', 'training.run_qwen3vl_poleval_qlora',
            '--execute', '--corpus', str(corpus),
            '--output', str(work / 'training')], 'train', cwd=repo)
dev = json.loads((work / 'training' / 'dev-metrics.json').read_text(encoding='utf-8'))
print('DEV_METRICS', json.dumps(dev, ensure_ascii=False)[:600])


In [ ]:
# Oficjalny scoring DEV: PolEvalDS/evaluation, protokol A (1-CER z CI).
import csv
expected = work / 'official-scoring' / 'expected.tsv'
outp = work / 'official-scoring' / 'out.tsv'
expected.parent.mkdir(parents=True, exist_ok=True)
rows = [json.loads(x) for x in (corpus / 'pages.jsonl').read_text(encoding='utf-8').splitlines()]
dev_refs = {r['id']: r['text'] for r in rows if r['split'] == 'validation'}
preds = {json.loads(x)['id']: json.loads(x) for x in (work / 'training' / 'dev-epoch-best-predictions.jsonl').read_text(encoding='utf-8').splitlines()} if (work / 'training' / 'dev-epoch-best-predictions.jsonl').exists() else {}
# Fallback: ostatnie predykcje epokowe wybrane przez ledger
if not preds:
    import glob
    files = sorted(glob.glob(str(work / 'training' / 'dev-epoch*-predictions.jsonl')))
    if not files:
        raise RuntimeError('Brak predykcji DEV do scoringu.')
    preds = {json.loads(x)['id']: json.loads(x) for x in Path(files[-1]).read_text(encoding='utf-8').splitlines()}
expected.write_text(''.join(f'{k}\t{json.dumps(v, ensure_ascii=False)}\n' for k, v in sorted(dev_refs.items())), encoding='utf-8')
outp.write_text(''.join(f'{k}\t{json.dumps(preds[k].get("text", ""), ensure_ascii=False)}\n' for k in sorted(dev_refs)), encoding='utf-8')
run_logged([sys.executable, str(corpus / 'evaluation' / 'polocr_eval.py'), '--subtask', 'A',
            '--expected', str(expected), '--out', str(outp)], 'official-score')


In [ ]:
# Dowody na Drive + reczne pobranie.
from google.colab import files
zip_path = work / 'training' / 'qwen3vl-poleval-qlora-v1-evidence.zip'
if not zip_path.is_file():
    raise RuntimeError('Brak koncowego ZIP-a. Czesciowe dowody sa na Drive: ' + str(work))
print('DOWODY_NA_DRIVE', zip_path)
files.download(str(zip_path))
